# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to explore and process the FAIR² dataset, a mass spectrometry record package with rich Croissant metadata. All entities are referenced by their `@id` as per the Croissant specification.

### Dataset Source
The dataset schema is available at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

> **Package DOI:** 10.71728/senscience.vq4a-28xa  
> **Title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show high-level metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Keywords: {getattr(metadata, 'keywords', None)}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}\n")

## 2. Data Overview
Explore the available record sets and their fields.

We use Croissant's `@id` system to reference all entities.

**List all record sets, fields, and columns (with their `@id`).**

In [ ]:
from pprint import pprint

# List all record sets
recordsets = list(dataset.record_sets)

print("Available Record Sets and their fields:")
for rs in recordsets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields:")
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
        print(f"    - {field_id}")
    columns = rs.get('column', [])
    if columns:
        if isinstance(columns, dict):
            columns = [columns]
        print(f"  Columns:")
        for column in columns:
            col_id = column['@id'] if isinstance(column, dict) and '@id' in column else column
            print(f"    - {col_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We load each record set found previously (identified by their `@id`).

In [ ]:
# If there are record sets, extract all.
record_set_ids = [rs['@id'] for rs in recordsets]
dataframes = {}

if not record_set_ids:
    print("No record sets found.")
else:
    for rs_id in record_set_ids:
        # Load records as list of dicts
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded record set '{rs_id}' with {len(df)} records. Columns: {df.columns.tolist()[:7]} ...")
        else:
            print(f"Record set '{rs_id}' is empty.")
# Pick the first loaded dataframe for demonstration
if dataframes:
    demo_rs = list(dataframes.keys())[0]
    demo_df = dataframes[demo_rs]
    print(f"\nColumns in {demo_rs}:\n", demo_df.columns.tolist())
    print(f"\nFirst 5 records:")
    display(demo_df.head())
else:
    print("No dataframes extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping, using column `@id`s.

> - **Numeric field**: Identify a column with numerical values such as `Coverage` or `MW` (molecular weight) by its `@id` and use it for filtering and normalization.
> - **Group field**: Identify a grouping attribute, such as `description` or `accession`, by its `@id` to aggregate the data.

In [ ]:
# Attempt basic EDA on our demonstration record set
import numpy as np

# Change these IDs to actual column '@id's as listed in Section 2 if necessary
if dataframes:
    numeric_fields = [col for col in demo_df.columns if demo_df[col].dtype in [np.float64, np.int64, float, int]]
    group_fields = [col for col in demo_df.columns if 'description' in col or 'accession' in col or 'group' in col.lower()]

    # Heuristically pick the first numeric field
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = demo_df[numeric_field].mean() if not np.isnan(demo_df[numeric_field].mean()) else 0

        filtered_df = demo_df[demo_df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (using field @id): {len(filtered_df)} records\n")
        print(filtered_df.head(3))

        # Normalize field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized values for {numeric_field}:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))

        # Group if a groupable field exists
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped by '{group_field}' (field @id), mean {numeric_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found in the demonstration record set.")
else:
    print("No data for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and relationships with categorical/grouping attributes.

We use matplotlib and seaborn for easy plotting. All field accesses here use the columns derived from their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_fields:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(demo_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field} (@id)')
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot per group (if group field exists)
    if group_fields:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=demo_df)
        plt.title(f'{numeric_field} by {group_field} (@id)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

else:
    print("No numeric data available for visualization.")

## 6. Conclusion
In this notebook, we leveraged the `mlcroissant` library to explore the FAIR² dataset structured by a rich Croissant schema. All processing referenced entities by their `@id`, adhering to reproducible research standards. After data extraction, EDA, and visualization, you can adapt further analysis by referencing the field and record set `@id`s printed above.

For more info, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/).